Python para ciência de dados - Trabalho de conclusão de materia

In [ ]:
#Equipe: 9

#Integrantes:

#Juliano Costa Silva
#Jandir Mendes Silva
#Matheus Pedroso
#João Vitor Lima do Amaral
#Thais da Costa Vicente

In [ ]:
#Importação de biblioteca#
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações para melhor visualização dos gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

# Definido funções úteis
def load_data(file_path):
    print(f"Carregando dados do arquivo CSV: {file_path}\n")
    data = pd.read_csv(file_path)

    print("Informações iniciais do DataFrame:\n")
    data.info()

    return data

def preprocess_data(df):
    print("\nIniciando pré-processamento dos dados...\n")

    #Transformação de tipologia#
    print("Convertendo o tipo da coluna \"pop\" de float64 para int64...")
    df['pop'] = df['pop'].astype('int64')

    # Removendo espaços em branco dos nomes das colunas
    df.columns = df.columns.str.strip()
    print("Removendo espaços em branco dos nomes das colunas...")

    #Remoção duplicados#
    print("Removendo linhas duplicadas do DataFrame...")
    df.drop_duplicates(inplace=True)

    #Remoção de nulos#
    print("Removendo linhas com valores nulos do DataFrame...")
    df.dropna(inplace=True)

    #Arredondando resultados#
    print("Arredondando os valores das colunas 'lifeExp' e 'gdpPercap'...")
    df['lifeExp'] = df['lifeExp'].round(2)
    df['gdpPercap'] = df['gdpPercap'].round(0)

    #Criação de coluna#
    print("Criando nova coluna 'log_gdpPercap' com o logaritmo de 'gdpPercap'...")
    df['log_gdpPercap'] = np.log(df['gdpPercap'])

    return df

print("📚 Bibliotecas necessárias importadas e sistema inicializado.")

In [ ]:
#Carregando dados#
print("📊 Carregando dados do arquivo CSV...")
df_gapminder = preprocess_data(pd.read_csv('gapminder_full.csv'))

In [ ]:
#Observação de resultado#
print("Visualizando as primeiras linhas do DataFrame após a remoção de espaços em branco:")
print(df_gapminder.head())

In [ ]:
#Obtendo as estatisticas descritivas#
print("\nEstatísticas descritivas do DataFrame:")
print(df_gapminder.describe())

#Total de linhas#
print("\nTotal de linhas no DataFrame após limpeza:", len(df_gapminder))

#Linhas únicas#
print("\nNúmero de países únicos no DataFrame:", df_gapminder['country'].nunique())

In [ ]:
# Calculando a média global por ano
df_global_ano = df_gapminder.groupby('year').agg({
    'lifeExp': 'mean',
    'gdpPercap': 'mean',
    'pop': 'sum'
}).reset_index()

# Plotando a evolução da expectativa de vida
ax = plt.gca()
sns.lineplot(data=df_global_ano, x='year', y='lifeExp', marker='o', ax=ax)
ax.set_title('Evolução da Expectativa de Vida Média Global (1952-2007)')
ax.set_xlabel('Ano')
ax.set_ylabel('Expectativa de Vida (Média)')
plt.show()

In [ ]:
#  Média Global da Expectativa de Vida ao Longo do Tempo
life_exp_time = df_gapminder.groupby('year')['lifeExp'].mean()

life_exp_time.plot(title='Média Global da Expectativa de Vida ao Longo do Tempo')
plt.ylabel('Expectativa de Vida Média')
plt.xlabel('Ano')
plt.grid(axis='y', linestyle='--')
plt.savefig('analise_1_life_exp_time.png')

In [ ]:
#  Distribuição do PIB per Capita
plt.figure(figsize=(10, 5))
plt.hist(np.log10(df_gapminder['gdpPercap']), bins=30, edgecolor='black')
plt.title('Distribuição do PIB per Capita (escala log base 10)')
plt.xlabel('log10(PIB per Capita)')
plt.ylabel('Frequência')
plt.savefig('analise_2_gdp_distribution.png')

In [ ]:
# Top 10 Países por PIB per Capita no Último Ano (2007)
df_2007 = df_gapminder[df_gapminder['year'] == 2007]
top_10_gdp = df_2007.sort_values(by='gdpPercap', ascending=False).head(10)

print("Top 10 Países por PIB per Capita em 2007:")
print(top_10_gdp[['country', 'gdpPercap']])

In [ ]:
# Correlação entre Expectativa de Vida e PIB per Capita
correlation = df_gapminder['lifeExp'].corr(df_gapminder['gdpPercap'])

print(f"Correlação Pearson entre Expectativa de Vida e PIB per Capita (todos os anos): {correlation:.4f}")

In [ ]:
#  Box Plot da Expectativa de Vida por Continente
df_gapminder.boxplot(column='lifeExp', by='continent', figsize=(10, 6))
plt.title('Distribuição da Expectativa de Vida por Continente')
plt.suptitle('')
plt.ylabel('Expectativa de Vida')
plt.xlabel('Continente')
plt.savefig('analise_5_life_exp_continent_boxplot.png')

In [ ]:
# População Total Global ao Longo do Tempo
total_pop = df_gapminder.groupby('year')['pop'].sum()

plt.figure(figsize=(10, 5))
total_pop.plot(title='População Total Global ao Longo do Tempo (em bilhões)')
plt.ylabel('População (x10^9)')
plt.xlabel('Ano')
plt.ticklabel_format(style='plain', axis='y')
plt.savefig('analise_6_global_pop_time.png')

In [ ]:
# Contagem de Países Únicos por Continente
country_count = df_gapminder.groupby('continent')['country'].nunique().sort_values(ascending=False)

country_count.plot(kind='bar', title='Contagem de Países Únicos por Continente')
plt.ylabel('Número de Países')
plt.xlabel('Continente')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('analise_7_country_count_continent.png')

In [ ]:
# Média da Expectativa de Vida por Continente (Média Geral)
mean_life_exp_continent = df_gapminder.groupby('continent')['lifeExp'].mean().sort_values(ascending=False)

print("Média Geral da Expectativa de Vida por Continente:")
print(mean_life_exp_continent)

In [ ]:
# País com o Maior Aumento Absoluto de População em um Intervalo
df_gapminder['pop_change'] = df_gapminder.groupby('country')['pop'].diff()
max_pop_increase = df_gapminder.loc[df_gapminder['pop_change'].idxmax()]

print("Maior Aumento Absoluto de População Registrado:")
print(f"País: {max_pop_increase['country']}")
print(f"Ano: {max_pop_increase['year']} (em relação ao ano anterior)")
print(f"Aumento: {max_pop_increase['pop_change']:,.0f}")
# Remove a coluna temporária
df_gapminder.drop(columns=['pop_change'], inplace=True)

In [ ]:
# País com a Menor Expectativa de Vida Registrada
min_life_exp = df_gapminder.loc[df_gapminder['lifeExp'].idxmin()]

print("Menor Expectativa de Vida Registrada:")
print(f"País: {min_life_exp['country']}")
print(f"Ano: {min_life_exp['year']}")
print(f"Expectativa de Vida: {min_life_exp['lifeExp']:.3f}")

In [ ]:
# Análise da Expectativa de Vida para um Continente Específico (Ex: 'Africa')
df_africa = df_gapminder[df_gapminder['continent'] == 'Africa']
mean_life_africa = df_africa.groupby('year')['lifeExp'].mean()

mean_life_africa.plot(title='Média da Expectativa de Vida na África ao Longo do Tempo')
plt.ylabel('Expectativa de Vida Média')
plt.xlabel('Ano')
plt.savefig('analise_11_africa_life_exp_time.png')

In [ ]:
# Scatter Plot de Expectativa de Vida vs PIB per Capita (Colorido por Continente)
sns.scatterplot(x='gdpPercap', y='lifeExp', hue='continent', data=df_gapminder)
plt.xscale('log')
plt.title('Expectativa de Vida vs PIB per Capita por Continente')
plt.xlabel('PIB per Capita (Escala Logarítmica)')
plt.ylabel('Expectativa de Vida')
plt.legend(title='Continente')
plt.savefig('analise_12_scatter_life_gdp_continent.png')

In [ ]:
# Crescimento Percentual do PIB per Capita (Comparando 2007 com 1952)
gdp_1952 = df_gapminder[df_gapminder['year'] == 1952][['country', 'gdpPercap']].set_index('country')
gdp_2007 = df_gapminder[df_gapminder['year'] == 2007][['country', 'gdpPercap']].set_index('country')

gdp_change = ((gdp_2007 - gdp_1952) / gdp_1952) * 100
gdp_change.columns = ['gdp_growth_pct']
gdp_change = gdp_change.sort_values(by='gdp_growth_pct', ascending=False).dropna()

print("Top 5 Crescimentos Percentuais de PIB per Capita (2007 vs 1952):")
print(gdp_change.head(5))

In [ ]:
# Estatísticas Descritivas para Variáveis Numéricas
desc_stats = df_gapminder[['lifeExp', 'gdpPercap', 'pop']].describe()

print("Estatísticas Descritivas (Expectativa de Vida, PIB per Capita, População):")
print(desc_stats)

In [ ]:
# Países que Alcançaram uma Alta Expectativa de Vida Consistente
# Verifica se o mínimo da lifeExp por país é maior que 70
consistent_high_life = df_gapminder.groupby('country')['lifeExp'].min()
consistent_countries = consistent_high_life[consistent_high_life >= 70].index.tolist()

print("Países com Expectativa de Vida Mínima (em qualquer ano) maior ou igual a 70 anos:")
print(consistent_countries)

In [ ]:
# Criação de uma Nova Coluna para o PIB Total
df_gapminder['total_gdp'] = df_gapminder['gdpPercap'] * df_gapminder['pop']

# Exibe os 5 maiores PIB's Totais no último ano (2007)
df_2007_gdp = df_gapminder[df_gapminder['year'] == 2007].sort_values(by='total_gdp', ascending=False).head(5)

print("Top 5 Países por PIB Total (2007):")
print(df_2007_gdp[['country', 'total_gdp']])
# Remove a coluna temporária
df_gapminder.drop(columns=['total_gdp'], inplace=True)

In [ ]:
# Comparação dos 5 Países Mais Populosos (1952 vs 2007)
top_5_1952 = df_gapminder[df_gapminder['year'] == 1952].sort_values(by='pop', ascending=False).head(5)[['country', 'pop']]
top_5_2007 = df_gapminder[df_gapminder['year'] == 2007].sort_values(by='pop', ascending=False).head(5)[['country', 'pop']]

print("Top 5 Países Mais Populosos em 1952:")
print(top_5_1952)

print("\nTop 5 Países Mais Populosos em 2007:")
print(top_5_2007)

In [ ]:
# Análise de Frequência dos Pontos no Tempo
year_counts = df_gapminder['year'].value_counts().sort_index()
num_years = df_gapminder['year'].nunique()

print(f"Número de Pontos de Tempo (Anos) no Dataset: {num_years}")
print("\nContagem de Registros por Ano:")
print(year_counts)

In [ ]:
# Amplitude Interquartil (IQR) para o PIB per Capita por Continente (2007)
df_2007 = df_gapminder[df_gapminder['year'] == 2007]
gdp_iqr = df_2007.groupby('continent')['gdpPercap'].agg(lambda x: x.quantile(0.75) - x.quantile(0.25))

print("Amplitude Interquartil (IQR) do PIB per Capita por Continente (2007):")
print(gdp_iqr.sort_values(ascending=False))

In [ ]:
# Visualização da Mudança na Expectativa de Vida para um País Específico (Ex: 'Brazil')
country_name = 'Brazil'
df_country = df_gapminder[df_gapminder['country'] == country_name]

plt.plot(df_country['year'], df_country['lifeExp'], marker='o')
plt.title(f'Expectativa de Vida no(a) {country_name} ao Longo do Tempo')
plt.xlabel('Ano')
plt.ylabel('Expectativa de Vida')
plt.grid(True)
plt.savefig(f'analise_20_life_exp_{country_name}.png')